In [7]:
from optclaw.config import get_app_config
from optclaw.models import create_chat_model
from langchain.agents import create_agent

In [8]:
from optclaw.agents.thread_state import ThreadDataState
from optclaw.agents.middlewares.thread_data_middleware import ThreadDataMiddleware
from optclaw.agents.middlewares.uploads_middleware import UploadsMiddleware

In [9]:
from optclaw.skills import get_skills_root_path
get_skills_root_path()

WindowsPath('E:/智能体/optclaw/skills')

In [4]:
path1 = '/mnt/skills/public/surprise-me/SKILL.md'
path2 = '/mnt/user-data/'

In [5]:
from optclaw.config import get_paths, get_app_config

# 1. 只获取一次配置（安全、高效）
paths = get_paths()
app_config = get_app_config()

# 2. 基础目录
root_dir = paths.base_dir.parent.parent
print(root_dir)

# 3. 容器路径前缀
container_prefix = app_config.skills.container_path

# 4. 安全切割路径（确保以 prefix 开头）
if path1.startswith(container_prefix):
    relative_path = path1[len(container_prefix):]
else:
    relative_path = path1  # 兜底，避免错误

# 5. 拼接最终路径（最安全）
full_path = root_dir / "skills" / relative_path.lstrip("/aas")  # 去掉开头斜杠，避免路径错乱

print(full_path.resolve())

In [6]:
container_skill_path = get_app_config().skills.container_path
actual_path = get_paths().base_dir.parent.parent / "skills" / path1[len(container_skill_path):].lstrip("/")
actual_path

WindowsPath('E:/智能体/optclaw/skills/public/surprise-me/SKILL.md')

In [7]:
import os
is_readonly = not os.access(full_path, os.W_OK)
is_readonly

True

In [8]:
from optclaw.tools.builtins import write_file_tool

In [ ]:
if os.path.exists(actual_path) and not os.access(actual_path, os.W_OK):
    print(f"Path {actual_path} is read-only.")

Path E:\智能体\optclaw\skills\public\surprise-me\SKILL.md is read-only.


In [11]:
path4 = '/mnt/skills/public/surprise-me/test.md'

In [12]:
write_file_tool(path4, "Hello, OptClaw!")

'Hello, OptClaw!'

# SKILL(本质是提示词)

In [22]:
# 设置技能文件开启标记，加载哪些技能基本信息
from optclaw.config import get_extensions_config
get_extensions_config()

ExtensionsConfig(mcp_servers={'filesystem': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '/path/to/allowed/files'], env={}, url=None, headers={}, oauth=None, description='Provides filesystem access within allowed directories'), 'github': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-github'], env={'GITHUB_TOKEN': ''}, url=None, headers={}, oauth=None, description='GitHub MCP server for repository operations'), 'postgres': McpServerConfig(enabled=False, type='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-postgres', 'postgresql://localhost/mydb'], env={}, url=None, headers={}, oauth=None, description='PostgreSQL database access')}, skills={'academic-paper-review': SkillStateConfig(enabled=False)})

In [23]:
# 技能加载
from optclaw.agents.pormpt_manager import prime_enabled_skills_cache
prime_enabled_skills_cache()

In [24]:
from optclaw.agents.pormpt_manager import get_skills_prompt_section,_get_enabled_skills

In [25]:
get_skills_prompt_section()

'<skill_system>\nYou have access to skills that provide optimized workflows for specific tasks. Each skill contains best practices, frameworks, and references to additional resources.\n\n**Progressive Loading Pattern:**\n1. When a user query matches a skill\'s use case, immediately call `read_file` on the skill\'s main file using the path attribute provided in the skill tag below\n2. Read and understand the skill\'s workflow and instructions\n3. The skill file contains references to external resources under the same folder\n4. Load referenced resources only when needed during execution\n5. Follow the skill\'s instructions precisely\n\n**Skills are located at:** /mnt/skills\n\n<available_skills>\n    <skill>\n        <name>bootstrap</name>\n        <description>Generate a personalized SOUL.md through a warm, adaptive onboarding conversation. Trigger when the user wants to create, set up, or initialize their AI partner\'s identity — e.g., "create my SOUL.md", "bootstrap my agent", "set u

# 工具（mcp+tools）

In [1]:
from optclaw.tools import get_available_tools
tools = get_available_tools()

format='2026-05-12 11:29:00,913 - optclaw.tools.tools - get_available_tools - INFO - Tool configs after group filtering: ['web_search', 'web_fetch', 'image_search']'
format='2026-05-12 11:29:01,036 - optclaw.tools.tools - get_available_tools - INFO - Total tools loaded: 3, built-in tools: 6'


In [2]:
tools

[StructuredTool(name='web_search', description='Search the web for information. Use this tool to find current information, news, articles, and facts from the internet.', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search_tool at 0x00000186B5879620>),
 StructuredTool(name='web_fetch', description='Fetch the contents of a web page at a given URL.\nOnly fetch EXACT URLs that have been provided directly by the user or have been returned in results from the web_search and web_fetch tools.\nThis tool can NOT access content that requires authentication, such as private Google Docs or pages behind login walls.\nDo NOT add www. to URLs that do NOT have them.\nURLs must include the schema: https://example.com is a valid URL while example.com is an invalid URL.', args_schema=<class 'langchain_core.utils.pydantic.web_fetch'>, coroutine=<function web_fetch_tool at 0x00000186B5879C60>),
 StructuredTool(name='image_search', description='Search for images online.

In [1]:
path4 = '/mnt/skills/public/surprise-me/SKILL.md'

In [2]:
from optclaw.tools.builtins import grep_file_tool,glob_file_tool

In [3]:
glob_file_tool(path4)

"Found 1 items matching pattern '/mnt/skills/public/surprise-me/':\n- surprise-me"

In [6]:
grep_file_tool(path4, "Step")

"Found 5 matches for 'Step' in /mnt/skills/public/surprise-me/SKILL.md:\nLine 12: ### Step 1: Discover Available Skills\nLine 16: ### Step 2: Plan the Surprise\nLine 33: ### Step 3: Fallback — No Other Skills Available\nLine 41: ### Step 4: Execute\nLine 48: ### Step 5: Reveal"

In [14]:
get_app_config().get_tool_config("web_search")

ToolConfig(name='web_search', group='web', use='optclaw.community.ddg_search.tools:web_search_tool', max_results=5)

In [1]:
from optclaw.tools.builtins import glob_file_tool

In [6]:
print(glob_file_tool("/mnt/skills/public/bootstrap/*.md"))

Found 1 items matching pattern '/mnt/skills/public/bootstrap/*.md':
- SKILL.md


# Agent

In [1]:
from optclaw.client import OptClawClient, StreamEvent

format='2026-05-19 23:41:10,771 - optclaw.config.subagents_config - load_subagents_config_from_dict - INFO - Subagents config loaded: default timeout=900s, default max_turns=None, per-agent overrides={'general-purpose': 'timeout=1800s, max_turns=160', 'coder': 'timeout=300s, max_turns=80'}'


In [2]:
occ = OptClawClient(agent_name="new", model_name='deepseek-v4-flash', subagent_enabled=False)

In [3]:
await occ.chat("你好，你有几类subagent", thread_id="221121110519zpfzpf")

format='2026-05-19 23:41:15,012 - optclaw.tools.tools - get_available_tools - INFO - Tool configs after group filtering: ['web_search']'
format='2026-05-19 23:41:15,024 - optclaw.tools.tools - get_available_tools - INFO - Total tools loaded: 1, built-in tools: 12'
format='2026-05-19 23:41:15,204 - optclaw.client - _ensure_agent - INFO - Agent created: agent_name=new, model=deepseek-v4-flash, thinking=False'
format='2026-05-19 23:41:15,706 - httpx - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"'
format='2026-05-19 23:41:17,755 - optclaw.agents.memory.queue - add - INFO - Memory update queued for thread 221121110519zpfzpf, queue size: 1'


'你好！你有这个问题反复问了很多次，让我给你一个明确、直接的答案：\n\n**我没有预设的 subagent（子代理）。**\n\n我是一个单一智能体，不存在任何预定义、内置的 subagent 分类。\n\n但如果你希望我**为你创建一个自定义的 agent**，这是完全可以的——通过提供一份 **SOUL.md**（定义该 agent 的身份、人格和行为），我可以用 `setup_agent` 工具帮你生成。\n\n如果你有任何想创建特定 agent 的想法，随时告诉我！'

In [ ]:
await occ.get_thread(thread_id="1121110519zpfzpf")

In [1]:
from optclaw.tools.builtins.execute_tool import execute_python_file_tool

execute_python_file_tool("/mnt/skills/public/academic-paper-review/test_run.py")

format='2026-05-18 14:18:21,316 - optclaw.config.subagents_config - load_subagents_config_from_dict - INFO - Subagents config loaded: default timeout=900s, default max_turns=None, per-agent overrides={'general-purpose': 'timeout=1800s, max_turns=160', 'coder': 'timeout=300s, max_turns=80'}'


"Execution of 'test_run.py' finished with exit code 0.\n\n--- Standard Output ---\nhello world\n"

In [ ]:
file_info = occ.upload_files(thread_id="22311new22123123", files=["本体业务模型.txt"])

In [ ]:
from optclaw.config import get_paths
get_paths().memory_file


WindowsPath('E:/智能体/optclaw/optclaw/.opt-claw/agents/test/memory.json')

In [1]:
from optclaw.tools.builtins import read_file_tool
from optclaw.tools.builtins.list_dir_tool import list_directory_tool

format='2026-05-17 20:42:56,824 - optclaw.config.subagents_config - load_subagents_config_from_dict - INFO - Subagents config loaded: default timeout=900s, default max_turns=None, per-agent overrides={'general-purpose': 'timeout=1800s, max_turns=160', 'coder': 'timeout=300s, max_turns=80'}'


In [2]:
list_directory_tool(path='/mnt/skills/public/academic-paper-review')

"Showing 1 of 1 items in 'academic-paper-review':\n[FILE] SKILL.md  (11.9 KB, Modified: 2026-04-14 16:14:19)"

In [1]:
# read_file_tool(path='/mnt/skills/public/bootstrap/test123.txt', max_tokens=100)

In [ ]:
# str_replace_tool(path='/mnt/skills/public/bootstrap/test123.txt', old_pattern='zpf', new_text='zpf1', is_regex=True)

format='2026-05-15 18:29:30,765 - optclaw.tools.builtins.str_replace_tool - str_replace_tool - INFO - Successfully replaced 5 occurrence(s) in test123.txt.'


"Success: Replaced 5 occurrence(s) of 'zpf' in file 'test123.txt'."

In [ ]:
import time
import concurrent.futures
import asyncio

# 创建单线程池
conversion_pool = concurrent.futures.ThreadPoolExecutor(max_workers=1)

# 异步任务，模拟耗时操作
async def task():
    await asyncio.sleep(5)
    return "ok"

def test1():
    return asyncio.run(task())  # 创建异步任务

# 提交异步任务到线程池
future = conversion_pool.submit(test1)

print("主线程继续往下走，不卡！")

# 1. 调用 .result() 就会阻塞，等任务结束
result = future.result()
print(result)

# 关闭线程池
conversion_pool.shutdown(wait=True)

主线程继续往下走，不卡！
ok


In [7]:
from optclaw.agents.pormpt_manager import apply_prompt_template
apply_prompt_template(
    agent_name="zpf",
    available_skills=['bootstrap'])

E:\智能体\optclaw\optclaw\.opt-claw\agents\zpf\memory.json


'\n<role>\nYou are zpf, an open-source super agent.\n</role>\n\n\n\n\n<thinking_style>\n- Think concisely and strategically about the user\'s request BEFORE taking action\n- Break down the task: What is clear? What is ambiguous? What is missing?\n- **PRIORITY CHECK: If anything is unclear, missing, or has multiple interpretations, you MUST ask for clarification FIRST - do NOT proceed with work**\n</thinking_style>\n\n<clarification_system>\n**WORKFLOW PRIORITY: CLARIFY → PLAN → ACT**\n1. **FIRST**: Analyze the request in your thinking - identify what\'s unclear, missing, or ambiguous\n2. **SECOND**: If clarification is needed, call `ask_clarification` tool IMMEDIATELY - do NOT start working\n3. **THIRD**: Only after all clarifications are resolved, proceed with planning and execution\n\n**CRITICAL RULE: Clarification ALWAYS comes BEFORE action. Never start working and clarify mid-execution.**\n\n**MANDATORY Clarification Scenarios - You MUST call ask_clarification BEFORE starting work 

In [8]:
from optclaw.config import get_app_config

config = get_app_config()
print(config.skills.get_skills_path())

E:\智能体\optclaw\skills
